In [0]:
import requests
from pyspark.sql import Row
from pyspark.sql.functions import *

BASE_URL = "https://terrabrasilis.dpi.inpe.br/queimadas/geoserver/wfs"

params = {
    "service": "WFS",
    "version": "2.0.0",
    "request": "GetFeature",
    "typeName": "dados_abertos:focos_48h_br_todosats",
    "outputFormat": "application/json"
}

payload = requests.get(
    BASE_URL,
    params=params,
    timeout=300
).json()

features = payload["features"]

In [0]:
records = []

for feature in features:

    p = feature["properties"]

    records.append({
        "id_foco_bdq": p.get("id_foco_bdq"),
        "foco_id": p.get("foco_id"),
        "longitude": p.get("longitude"),
        "latitude": p.get("latitude"),
        "data_hora_gmt": p.get("data_hora_gmt"),
        "data_pas": p.get("data_pas"),
        "satelite": p.get("satelite"),
        "municipio": p.get("municipio"),
        "estado": p.get("estado"),
        "pais": p.get("pais"),
        "precipitacao": p.get("precipitacao"),
        "numero_dias_sem_chuva": p.get("numero_dias_sem_chuva"),
        "risco_fogo": p.get("risco_fogo"),
        "bioma": p.get("bioma"),
        "vegetacao": p.get("vegetacao"),
        "grade_wrs": p.get("grade_wrs"),
        "frp": p.get("frp")
    })

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("id_foco_bdq", LongType(), True),
    StructField("foco_id", StringType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("data_hora_gmt", StringType(), True),
    StructField("data_pas", StringType(), True),
    StructField("satelite", StringType(), True),
    StructField("municipio", StringType(), True),
    StructField("estado", StringType(), True),
    StructField("pais", StringType(), True),
    StructField("precipitacao", DoubleType(), True),
    StructField("numero_dias_sem_chuva", IntegerType(), True),
    StructField("risco_fogo", DoubleType(), True),
    StructField("bioma", StringType(), True),
    StructField("vegetacao", StringType(), True),
    StructField("grade_wrs", StringType(), True),
    StructField("frp", DoubleType(), True)
])

In [0]:
bronze_df = spark.createDataFrame(
    records,
    schema=schema
)

bronze_df = (
    bronze_df
    .withColumn(
        "ingestion_ts",
        current_timestamp()
    )
    .withColumn(
        "source_system",
        lit("INPE")
    )
    .withColumn(
        "source_layer",
        lit("dados_abertos:focos_48h_br_todosats")
    )
)

(
    bronze_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "gs_carbon.bronze.inpe_focos"
    )
)

In [0]:
%sql
SELECT * FROM gs_carbon.bronze.inpe_focos